In [ ]:
!pip install pymongo dnspython pandas numpy --quiet

In [ ]:
import requests

try:
    response = requests.get('https://api.ipify.org')
    public_ip = response.text
    print(f"Colab Runtime Public IP Address: {public_ip}")
except requests.exceptions.RequestException as e:
    print(f"Could not retrieve public IP address: {e}")


Colab Runtime Public IP Address: 34.26.63.205


In [ ]:
from __future__ import annotations

import argparse
import json
import logging
import math
from pathlib import Path

In [ ]:
DEFAULT_FIRES_JSON   = Path("/content/fires_raw_cleaned_fixed.json")
DEFAULT_WEATHER_CSV  = Path("/content/weather_covariates.csv")
DEFAULT_OUT_CSV      = Path("data/fires_final.csv")
LOG_PATH             = Path("logs/03_join_and_load.log")

MONGO_URI   = "mongodb+srv://<username>:<password>@cluster0.ikxi8.mongodb.net/?appName=Cluster0"
DB_NAME     = "wildfire_project"
COLLECTION  = "wildfires"

# Number of documents per MongoDB bulk_write batch
BATCH_SIZE  = 500

# Containment duration class boundaries (days)
DURATION_CLASSES = [
    (0,   1,   "< 1 day"),
    (1,   7,   "1-7 days"),
    (7,   30,  "8-30 days"),
    (30,  float("inf"), "> 30 days"),
]

In [ ]:
def setup_logging(log_path: Path) -> logging.Logger:
    """Configure file + console logging and return the module logger."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s  %(levelname)s  %(message)s",
        handlers=[
            logging.FileHandler(log_path, mode="w"),
            logging.StreamHandler(),
        ],
    )
    return logging.getLogger(__name__)

In [ ]:
def assign_duration_class(days: float | None) -> str | None:
    """
    Map a numeric containment_days value to one of the four ordered class
    labels used as classification targets in the ML model.

    Returns None when days is None (unknown containment date).
    """
    if days is None:
        return None
    for lo, hi, label in DURATION_CLASSES:
        if lo <= days < hi:
            return label
    return "> 30 days"


def rolling_kbdi(group: pd.DataFrame, fc_mm: float = 203.2) -> pd.Series:
    """
    Compute the Keetch-Byram Drought Index (KBDI) as a rolling daily
    cumulative value for a single spatial group (fires sorted by date).

    The Nelson (1980) formulation is used:
        dQ_i = (800 - Q_{i-1}) * (0.968 * exp(0.0486 * T_i) - 8.3)
               / (1 + 10.88 * exp(-0.0441 * P_i)) * 1e-3
    where T is max daily temperature (°F) and P is 30-day cumulative
    precipitation (inches).  We approximate P as the daily precip value
    where the full 30-day rolling sum is not available.

    Args:
        group:  DataFrame slice for a single (state, gacc_region) group,
                sorted ascending by discovery_date, with columns:
                max_temp_c, precip_mm.
        fc_mm:  Soil field capacity in mm (default 203.2 mm ≈ 8 inches).

    Returns:
        Series of KBDI values aligned to group.index.
    """
    kbdi_vals = []
    q = 0.0   # start at 0 (fully saturated) for each spatial group

    for _, row in group.iterrows():
        temp_c   = row.get("max_temp_c")
        precip   = row.get("precip_mm") or 0.0

        if temp_c is None or math.isnan(temp_c):
            kbdi_vals.append(None)
            continue

        # Convert to imperial for the Nelson formula
        temp_f    = temp_c * 9 / 5 + 32
        precip_in = precip / 25.4   # mm → inches

        if temp_f > 32:  # only evaporate above freezing
            numerator   = (800 - q) * (0.968 * math.exp(0.0486 * temp_f) - 8.3)
            denominator = 1 + 10.88 * math.exp(-0.0441 * precip_in)
            dq          = max(0.0, numerator / denominator * 1e-3)
        else:
            dq = 0.0

        # Precipitation reduces KBDI (rainfall above 0.2 in resets deficit)
        if precip_in > 0.2:
            q = max(0.0, q - (precip_in - 0.2) * 25.4 * 3.94)   # empirical discharge

        q = min(800.0, q + dq)
        kbdi_vals.append(round(q, 1))

    return pd.Series(kbdi_vals, index=group.index)

In [ ]:
def load_fires_json(fires_json: Path, logger: logging.Logger) -> pd.DataFrame:
    """
    Load the NDJSON fire records into a pandas DataFrame.

    Args:
        fires_json: Path to fires_raw.json.
        logger:     Module logger.

    Returns:
        DataFrame with one row per fire and columns matching the soft schema.
    """
    if not fires_json.exists():
        raise FileNotFoundError(f"Fires JSON not found: {fires_json}")

    records = []
    try:
        with fires_json.open("r", encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    doc = json.loads(line)
                    # Flatten the GeoJSON location sub-document for CSV
                    coords = doc.get("location", {}).get("coordinates", [None, None])
                    doc["longitude"] = coords[0]
                    doc["latitude"]  = coords[1]
                    doc.pop("location", None)
                    doc.pop("weather", None)   # weather comes from the CSV
                    records.append(doc)
    except (json.JSONDecodeError, OSError) as exc:
        logger.error("Failed to load fires JSON: %s", exc)
        raise

    df = pd.DataFrame(records).set_index("_id")
    logger.info("Loaded %d fire records", len(df))
    return df


def load_weather_csv(weather_csv: Path, logger: logging.Logger) -> pd.DataFrame:
    """
    Load the ERA5 weather covariates CSV written by 02_fetch_era5_weather.py.

    Args:
        weather_csv: Path to weather_covariates.csv.
        logger:      Module logger.

    Returns:
        DataFrame indexed by fire _id with weather columns.
    """
    if not weather_csv.exists():
        raise FileNotFoundError(f"Weather CSV not found: {weather_csv}")

    try:
        df = pd.read_csv(weather_csv, index_col="_id")
    except Exception as exc:
        logger.error("Failed to load weather CSV: %s", exc)
        raise

    logger.info("Loaded weather covariates for %d fires", len(df))
    return df

In [ ]:
def join_and_engineer(
    df_fires: pd.DataFrame,
    df_weather: pd.DataFrame,
    logger: logging.Logger,
) -> pd.DataFrame:
    """
    Left-join fire records with weather covariates on fire _id, then
    compute derived features:
      - Rolling KBDI per (state, fire_year) group
      - duration_class label for the ML classification target
      - containment_days for fires where both dates are present

    Args:
        df_fires:   Fire records DataFrame (indexed by _id).
        df_weather: Weather covariates DataFrame (indexed by _id).
        logger:     Module logger.

    Returns:
        Merged, feature-engineered DataFrame ready for CSV export and MongoDB.
    """
    # Left join — every fire kept even if weather fetch failed
    df = df_fires.join(df_weather, how="left", rsuffix="_weather")
    logger.info("After join: %d rows, %d columns", len(df), len(df.columns))

    # Compute rolling KBDI grouped by state × year, sorted by discovery_date
    logger.info("Computing rolling KBDI ...")
    df["discovery_date"] = pd.to_datetime(df["discovery_date"], errors="coerce", utc=True)
    df = df.sort_values("discovery_date")

    kbdi_series = (
        df.groupby(["state", "fire_year"], group_keys=False)
          .apply(rolling_kbdi)
    )
    # rolling_kbdi returns a Series — align back by index
    df["kbdi"] = kbdi_series.reindex(df.index)

    # Add the ML classification label for containment duration
    df["duration_class"] = df["containment_days"].apply(assign_duration_class)

    # Convert discovery_date back to ISO string for JSON serialisation
    df["discovery_date"] = df["discovery_date"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

    logger.info("Feature engineering complete")
    logger.info(
        "  duration_class distribution:\n%s",
        df["duration_class"].value_counts(dropna=False).to_string()
    )
    return df

In [ ]:
def upsert_to_mongo(
    df: pd.DataFrame,
    mongo_uri: str,
    db_name: str,
    collection_name: str,
    logger: logging.Logger,
) -> int:
    """
    Upsert all rows of df into the MongoDB Atlas collection as documents
    that conform to the wildfire soft schema.  Uses bulk_write with
    ReplaceOne(upsert=True) for efficiency.

    Args:
        df:              Feature-engineered DataFrame indexed by _id.
        mongo_uri:       MongoDB Atlas connection string.
        db_name:         Target database name.
        collection_name: Target collection name.
        logger:          Module logger.

    Returns:
        Total number of documents upserted (inserted + updated).
    """
    try:
        client     = MongoClient(mongo_uri, serverSelectionTimeoutMS=10_000)
        client.admin.command("ping")
        col        = client[db_name][collection_name]
        logger.info("Connected to MongoDB Atlas — %s.%s", db_name, collection_name)
    except pymongo.errors.ConnectionFailure as exc:
        logger.error("MongoDB connection failed: %s", exc)
        raise

    total_upserted = 0
    buffer         = []

    for fid, row in df.iterrows():
        # Build the weather sub-document (None if all fields are NaN)
        weather_doc = {
            "max_temp_c":         row.get("max_temp_c"),
            "wind_speed_ms":      row.get("wind_speed_ms"),
            "relative_humidity":  row.get("relative_humidity"),
            "kbdi":               row.get("kbdi"),
            "erc":                row.get("erc"),
        }
        has_weather = any(v is not None and not (isinstance(v, float) and math.isnan(v))
                          for v in weather_doc.values())

        # Rebuild the GeoJSON location sub-document
        lon = row.get("longitude")
        lat = row.get("latitude")
        location_doc = (
            {"type": "Point", "coordinates": [lon, lat]}
            if lon is not None and lat is not None else None
        )

        # Assemble the canonical document matching the soft schema
        doc = {
            "_id":              fid,
            "fire_name":        row.get("fire_name"),
            "fire_year":        int(row["fire_year"]) if pd.notna(row.get("fire_year")) else None,
            "discovery_date":   row.get("discovery_date"),
            "containment_date": row.get("containment_date"),
            "final_size_acres": float(row["final_size_acres"]) if pd.notna(row.get("final_size_acres")) else None,
            "containment_days": float(row["containment_days"]) if pd.notna(row.get("containment_days")) else None,
            "size_log":         float(row["size_log"]) if pd.notna(row.get("size_log")) else None,
            "size_class":       row.get("size_class"),
            "duration_class":   row.get("duration_class"),   # ML target label
            "ignition_cause":   row.get("ignition_cause"),
            "location":         location_doc,
            "state":            row.get("state"),
            "gacc_region":      row.get("gacc_region") or None,
            "weather":          weather_doc if has_weather else None,
            "fuel_model":       row.get("fuel_model") or None,
            "incident_type":    int(row["incident_type"]) if pd.notna(row.get("incident_type")) else None,
        }

        buffer.append(ReplaceOne({"_id": fid}, doc, upsert=True))

        # Flush when batch is full
        if len(buffer) >= BATCH_SIZE:
            total_upserted += _flush_batch(buffer, col, logger)
            buffer = []

    # Flush remaining documents
    if buffer:
        total_upserted += _flush_batch(buffer, col, logger)

    client.close()
    logger.info("MongoDB upsert complete — total upserted: %d", total_upserted)
    return total_upserted


def _flush_batch(
    ops: list,
    col: pymongo.collection.Collection,
    logger: logging.Logger,
) -> int:
    """
    Execute a bulk_write batch and return the count of affected documents.
    Logs individual write errors without raising so the pipeline continues.

    Args:
        ops:    List of ReplaceOne operations.
        col:    Target pymongo Collection.
        logger: Module logger.

    Returns:
        Number of documents upserted + matched in this batch.
    """
    try:
        result = col.bulk_write(ops, ordered=False)
        n = result.upserted_count + result.matched_count
        logger.info("  Batch of %d → upserted=%d matched=%d",
                    len(ops), result.upserted_count, result.matched_count)
        return n
    except pymongo.errors.BulkWriteError as bwe:
        for err in bwe.details.get("writeErrors", []):
            logger.warning("BulkWrite error: %s", err)
        return 0
    except Exception as exc:
        logger.error("Unexpected batch flush error: %s", exc)
        return 0

In [ ]:
import json
from pathlib import Path

file_path = Path("/content/fires_raw_cleaned.json")

if not file_path.exists():
    print(f"Error: File not found at {file_path}")
else:
    print(f"Validating JSON file: {file_path}")
    with file_path.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                json.loads(line)
            except json.JSONDecodeError as e:
                print(f"Error on line {i+1}: {line.strip()}\n  {e}")
            except Exception as e:
                print(f"Unexpected error on line {i+1}: {line.strip()}\n  {e}")
    print("JSON validation complete.")

Validating JSON file: /content/fires_raw_cleaned.json
Error on line 831792: {"_id": "SWRA_SC_44836", "fire_name": "UNNAMED", "fire_year": 1997, "discovery_date": null, "containment_date": null, "final_size_acres": 3.0, "containment_days": null, "size_log": 0.60206, "size_class": "B", "ignition_cause": "Missing data/not speci
  Unterminated string starting at: line 1 column 228 (char 227)
JSON validation complete.


In [ ]:
import json
from pathlib import Path
import logging

# Setup basic logging for this script
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('fix_json_errors.log', mode='w')
    ]
)
logger = logging.getLogger(__name__)

input_file = Path("/content/fires_raw_cleaned.json")
output_file = Path("/content/fires_raw_cleaned_fixed.json")

def fix_truncated_json(input_path: Path, output_path: Path) -> None:
    if not input_path.exists():
        logger.error(f"Input file not found: {input_path}")
        return

    logger.info(f"Starting to fix JSON file: {input_path}")
    valid_lines_count = 0
    skipped_lines_count = 0

    with input_path.open('r', encoding='utf-8') as infile, \
         output_path.open('w', encoding='utf-8') as outfile:
        for i, line in enumerate(infile):
            try:
                # Attempt to load the JSON object
                doc = json.loads(line)
                # If successful, write it to the output file
                outfile.write(json.dumps(doc) + '\n')
                valid_lines_count += 1
            except json.JSONDecodeError as e:
                logger.warning(
                    f"Skipping malformed line {i+1} in {input_path}: {line.strip()[:100]}... - Error: {e}"
                )
                skipped_lines_count += 1
            except Exception as e:
                logger.error(
                    f"Unexpected error on line {i+1} in {input_path}: {line.strip()[:100]}... - Error: {e}"
                )
                skipped_lines_count += 1

    logger.info(f"JSON fixing complete. Valid lines: {valid_lines_count}, Skipped lines: {skipped_lines_count}")
    logger.info(f"Fixed JSON written to: {output_path}")

# Run the fix function
fix_truncated_json(input_file, output_file)


In [ ]:
def parse_args() -> argparse.Namespace:
    """Parse command-line arguments."""
    parser = argparse.ArgumentParser(
        description="Join fire and weather data, engineer features, and load to MongoDB."
    )
    parser.add_argument("--fires",   type=Path, default=DEFAULT_FIRES_JSON)
    parser.add_argument("--weather", type=Path, default=DEFAULT_WEATHER_CSV)
    parser.add_argument("--out",     type=Path, default=DEFAULT_OUT_CSV)
    parser.add_argument(
        "--dry-run", action="store_true",
        help="Skip MongoDB write; write CSV only"
    )
    # In a notebook environment, parse_args() can pick up notebook-specific args
    # that are not defined. We pass an empty list to prevent this.
    return parser.parse_args([])


def main() -> None:
    """Main entry point — orchestrates join, feature engineering, and load."""
    args   = parse_args()
    logger = setup_logging(LOG_PATH)
    logger.info("=== 03_join_and_load.py started ===")

    try:
        df_fires   = load_fires_json(args.fires, logger)

        # Conditionally load weather data
        if args.weather.exists():
            df_weather = load_weather_csv(args.weather, logger)
        else:
            logger.warning(
                "Weather CSV not found at %s. Proceeding without weather covariates.",
                args.weather
            )
            df_weather = pd.DataFrame(index=df_fires.index) # Create an empty DataFrame with same index

        df_final   = join_and_engineer(df_fires, df_weather, logger)

        # Write final CSV
        args.out.parent.mkdir(parents=True, exist_ok=True)
        df_final.to_csv(args.out)
        logger.info("Wrote fires_final.csv → %s  (%d rows)", args.out, len(df_final))

        # Upsert to MongoDB unless --dry-run
        if not args.dry_run:
            # Import MongoClient and ReplaceOne here, as they are only used within upsert_to_mongo
            # and might not be available globally if the user didn't run all cells.
            from pymongo import MongoClient
            from pymongo.operations import ReplaceOne
            n = upsert_to_mongo(df_final, MONGO_URI, DB_NAME, COLLECTION, logger)
            logger.info("MongoDB load complete — %d documents upserted", n)
        else:
            logger.info("--dry-run flag set; MongoDB write skipped")

    except Exception as exc:
        logger.error("Fatal error: %s", exc)
        raise

    logger.info("=== 03_join_and_load.py finished ===")


if __name__ == "__main__":
    # Ensure pandas and pymongo are imported before main() if it's run directly
    import pandas as pd
    import pymongo
    from pymongo import MongoClient
    from pymongo.operations import ReplaceOne # Added to ensure ReplaceOne is available

    main()

/tmp/ipykernel_52530/298179245.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(rolling_kbdi)
